# 10. Base scan (v14.4)

## Purpose

Framework hypothesis: if masses are "digits in the owner's number system", there exists a base `b` in which `log_b(m_i/m_j)` are close to integers.

Main question: **for which fermions**? By §8 of the article, leptons are "output" (structure), quarks are "input" (data). If the framework is correct, the base should exist only for leptons.

## Statistics

For each pair (i < j) of sorted masses: compute ratio `m_j/m_i`, take `log_b`, measure distance to nearest integer, average over all pairs.

- **Scale-invariant by construction**: `log_b(m_j/m_i)` unchanged under `m → λm`.
- Range of bases: **2..40** (39 values).

## MC framing

Monte Carlo by PDG uncertainties (N=500, seed=42) = **stability test against measurement errors**. **NOT statistical significance**. Significance comes only from the permutation test below.

## Caveats (accepted through review R11–R15)

1. **Post-selection.** Base 15 was FOUND by scanning these data. Fixed-base-15 p-value is invalidated by post-selection; only best-of-39 correction is legitimate.
2. **Koide is not independent.** Integer exponents give K=0.65591603; fractional parts tune K to 2/3. Base-15 deviation (0.0281) and Koide precision (6e-6) are **one piece of evidence, not two**.
3. **Fractions 1/32, 1/91 are post-hoc.** Near-misses exist for any value.
4. **Structural cancellation.** On bases 16 and 4, τ-error contributions cancel pairwise, so std stays at 10⁻⁹–10⁻⁸. On base 15 they align, giving std ~10⁻⁵ after the τ-error fix.

## τ error fix (R13)

τ-error corrected to **0.12 MeV** (PDG 2022); was 0.001 in v14.0. Change does not affect the deterministic scan; only MC std on base 15.

## Expected results

| Category | Best base | dev | Bases < 5% |
|---|---|---|---|
| Leptons | 15 | **0.0281** | 1 (only 15) |
| Quarks | 40 | 0.174 | 0 |
| All 9 | **40** | **0.2138** | 0 |

**Header note (R11 §3.1):** line "All 9: base 16, 14%" from v14.0 was incorrect — fixed to base 40, 21.4%.

## Output

- `10_base_scan.png` — figure (left: base scan; right: Koide consistency).
- `nb10_pvalue_results.txt` — single source for all placeholders: scan, MC, permutation test, look-elsewhere. All documents (README, article, QnA) filled from this file simultaneously.

## Reference

- **§9.3** of article: "Base scan" (this notebook).
- **§8**: "Structure vs data" — main conceptual output.
- **§11 of v14.1**: base-15 as exploratory (not established, p_abs = P_abs).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from datetime import datetime

# PDG 2022 masses (MeV) / Массы (МэВ) по PDG 2022
leptons = {
    'names': ['e', 'mu', 'tau'],
    'values': np.array([0.51099895, 105.6583755, 1776.86]),
    # tau: 0.12 MeV (PDG 2022). v14.0 had 0.001 — off by two orders; fixed in R13 §2.2.
    # tau: 0.12 МэВ (PDG 2022). В v14.0 было 0.001 — ошибка на два порядка; исправлено в R13 §2.2.
    'errors': np.array([1e-8, 1e-6, 0.12]),
}
quarks = {
    'names': ['u', 'd', 's', 'c', 'b', 't'],
    'values': np.array([2.16, 4.67, 93.4, 1270.0, 4180.0, 172760.0]),
    'errors': np.array([0.49, 0.48, 8.6, 20.0, 30.0, 300.0]),
}
fermions_9 = {
    'names': leptons['names'] + quarks['names'],
    'values': np.concatenate([leptons['values'], quarks['values']]),
    'errors': np.concatenate([leptons['errors'], quarks['errors']]),
}
categories = {'leptons': leptons, 'quarks': quarks, 'fermions_9': fermions_9}

BASES = list(range(2, 41))


def make_ratios(values):
    """All pairwise ratios m_j / m_i (j > i) for sorted values."""
    v = np.sort(values)
    ratios = []
    for i in range(len(v)):
        for j in range(i + 1, len(v)):
            ratios.append(v[j] / v[i])
    return np.array(ratios)


def deviation(ratios, b):
    """Mean |log_b(r) - round(log_b(r))|."""
    logs = np.log(ratios) / np.log(b)
    return np.mean(np.abs(logs - np.round(logs)))


def deviation_mc(values, errors, b, N=500, seed=42):
    """MC by PDG uncertainties. Stability, NOT significance. Returns (mean, std)."""
    rng = np.random.default_rng(seed)
    devs = []
    for _ in range(N):
        noisy = values + rng.normal(0, 1, len(values)) * errors
        if np.any(noisy <= 0):
            continue
        devs.append(deviation(make_ratios(noisy), b))
    return np.mean(devs), np.std(devs)


def koide(masses):
    """K = (Σm) / (Σ√m)²."""
    m = np.array(masses)
    return np.sum(m) / np.sum(np.sqrt(m))**2


print(f"Ready. BASES: {BASES[0]}..{BASES[-1]}, tau error = {leptons['errors'][2]} MeV")

In [ ]:
print("=" * 75)
print("BASE SCAN (2..40), MC by PDG errors (N=500) — stability test only")
print("=" * 75)

results = {}
for cat_name, cat in categories.items():
    results[cat_name] = []
    for b in BASES:
        m, s = deviation_mc(cat['values'], cat['errors'], b, N=500)
        results[cat_name].append((b, m, s))

# Top-5 tables (std in scientific notation — R11 §3.4) / Таблицы топ-5 (std в научной нотации)
for cat_name in categories:
    sorted_res = sorted(results[cat_name], key=lambda x: x[1])
    print(f"\n{cat_name.upper()} ({len(categories[cat_name]['values'])} masses)")
    print(f"  {'Rank':>4} {'Base':>5} {'Avg dev':>10} {'± std':>12}")
    print(f"  {'-' * 36}")
    for rank, (b, m, s) in enumerate(sorted_res[:5], 1):
        print(f"  {rank:>4} {b:>5} {m:>10.4f} ±{s:>10.4e}")

# Structural bases / Структурные базы
print("\n" + "=" * 75)
print("STRUCTURAL BASES (2^k - 1 and 2^k):")
print("=" * 75)
struct_bases = [2, 3, 4, 7, 8, 15, 16, 31, 32]
print(f"{'Base':>5} {'Formula':>10} {'Leptons':>10} {'Quarks':>10} {'All 9':>10}")
print(f"{'-' * 50}")
for b in struct_bases:
    formula = ""
    for k in range(1, 6):
        if b == 2**k:
            formula = f"2^{k}"
        if b == 2**k - 1:
            formula = f"2^{k}-1"
    rl = [r for r in results['leptons'] if r[0] == b][0]
    rq = [r for r in results['quarks'] if r[0] == b][0]
    rf = [r for r in results['fermions_9'] if r[0] == b][0]
    print(f"{b:>5} {formula:>10} {rl[1]:>10.4f} {rq[1]:>10.4f} {rf[1]:>10.4f}")

# Counters / Счётчики
n_good_lep = sum(1 for _, m, _ in results['leptons'] if m < 0.05)
n_good_q = sum(1 for _, m, _ in results['quarks'] if m < 0.05)
n_good_all = sum(1 for _, m, _ in results['fermions_9'] if m < 0.05)
print(f"\nBases with dev < 5%: leptons = {n_good_lep}, quarks = {n_good_q}, all 9 = {n_good_all}")
print("NOTE: base 15 is best for leptons (dev=0.0281), but this does NOT")
print("      establish significance — see permutation test below.")

In [ ]:
print("=" * 75)
print("KOIDE (K = 2/3): descriptive, post-hoc")
print("=" * 75)

m_e, m_mu, m_tau = leptons['values']
a_mu = np.log(m_mu / m_e) / np.log(15)
a_tau = np.log(m_tau / m_e) / np.log(15)
print(f"  a_mu  = log_15(m_mu/m_e)  = {a_mu:.10f}")
print(f"  a_tau = log_15(m_tau/m_e) = {a_tau:.10f}")

print(f"\nNearest 'nice' fractions (post-hoc):")
print(f"  a_mu  ≈ 2 - 1/32 = {2 - 1/32:.10f}  (err {abs(a_mu - (2 - 1/32)):.2e})")
print(f"  a_tau ≈ 3 + 1/91 = {3 + 1/91:.10f}  (err {abs(a_tau - (3 + 1/91)):.2e})")

K_real = koide([m_e, m_mu, m_tau])
K_int = koide([1.0, 15**2, 15**3])
K_frac = koide([1.0, 15**(2 - 1/32), 15**(3 + 1/91)])

print(f"\nKoide K:")
print(f"  Integer exponents (0,2,3): K = {K_int:.8f}  (dev {abs(K_int - 2/3):.2e})")
print(f"  Fractions (1/32, 1/91):    K = {K_frac:.8f}  (dev {abs(K_frac - 2/3):.2e})")
print(f"  Real masses:               K = {K_real:.8f}  (dev {abs(K_real - 2/3):.2e})")

print("\nNOTE: fractions are post-hoc — near-misses exist for any value.")
print("The 1e-5 precision comes from Koide for measured masses.")
print("Base-15 deviation (0.0281) and Koide are NOT independent pieces of evidence:")
print("integer exponents give K=0.656; fractional parts tune K to 2/3.")
print("Same evidence, two formulations.")

In [ ]:
print("=" * 75)
print("PERMUTATION TEST (84 splits, best-of-39, deterministic)")
print("=" * 75)


def dev_best39(values):
    r = make_ratios(values)
    return min(deviation(r, b) for b in BASES)


ALL9 = fermions_9['values']
n = len(ALL9)

triplet_devs = []
split_asyms = []
for tri in combinations(range(n), 3):
    tri_l = list(tri)
    six = [i for i in range(n) if i not in tri_l]
    d3 = dev_best39(ALL9[tri_l])
    d6 = dev_best39(ALL9[six])
    triplet_devs.append((tuple(tri), d3))
    split_asyms.append((tuple(tri), d6 - d3))

lep_idx = (0, 1, 2)  # e, mu, tau
obs_d_lep = dev_best39(leptons['values'])
obs_d_q = dev_best39(quarks['values'])
obs_asym = obs_d_q - obs_d_lep

p_abs_count = sum(1 for _, d in triplet_devs if d <= obs_d_lep + 1e-12)
p_asym_count = sum(1 for _, a in split_asyms if a >= obs_asym - 1e-12)

sorted_triplets = sorted(triplet_devs, key=lambda x: x[1])
rank_lep = next(i for i, (idx, _) in enumerate(sorted_triplets, 1) if idx == lep_idx)

print(f"  dev_best39(leptons) = {obs_d_lep:.6f}")
print(f"  dev_best39(quarks)  = {obs_d_q:.6f}")
print(f"  observed asym       = {obs_asym:.6f}")
print(f"  p_abs               = {p_abs_count}/84 = {p_abs_count / 84:.4f}")
print(f"  rank of leptons     = {rank_lep}/84")
print(f"  p_asym              = {p_asym_count}/84 = {p_asym_count / 84:.4f}")

if p_abs_count == 1:
    status = "established (discrete floor)"
elif p_abs_count <= 5:
    status = "suggestive"
else:
    status = "NOT established"
print(f"  Status: {status}  (suggestive floor: 5/84 = 0.0595)")

# Fixed base 15 (secondary, with caveat) / Фиксированная база 15 (вторично, с оговоркой)
obs_fixed = deviation(make_ratios(leptons['values']), 15)
p_fixed_count = sum(
    1 for tri in combinations(range(n), 3)
    if deviation(make_ratios(ALL9[list(tri)]), 15) <= obs_fixed + 1e-12
)
print(f"\n  Fixed base 15 (secondary, post-selection caveat):")
print(f"    p_fixed = {p_fixed_count}/84 = {p_fixed_count / 84:.4f}")

# Look-elsewhere / Эффект поиска в другом месте
top_idx, top_dev = sorted_triplets[0]
top_base = min(BASES, key=lambda b: deviation(make_ratios(ALL9[list(top_idx)]), b))
top_names = [fermions_9['names'][i] for i in top_idx]
print(f"\n  Look-elsewhere: top-1 triplet = {{{', '.join(top_names)}}}, "
      f"best base {top_base}, dev = {top_dev:.6f}")
print(f"  ratio = {obs_d_lep / top_dev:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: base scan / Слева: скан баз
colors = {'leptons': 'blue', 'quarks': 'red', 'fermions_9': 'green'}
labels = {'leptons': 'Leptons (3)', 'quarks': 'Quarks (6)', 'fermions_9': 'All 9'}
for cat_name in ['leptons', 'quarks', 'fermions_9']:
    bv = [r[0] for r in results[cat_name]]
    dv = [r[1] for r in results[cat_name]]
    sv = [r[2] for r in results[cat_name]]
    axes[0].errorbar(bv, dv, yerr=sv, fmt='o-', color=colors[cat_name],
                     label=labels[cat_name], ms=4, capsize=2, alpha=0.7)
for b in [3, 7, 15, 16]:
    axes[0].axvline(b, color='gray', ls=':', lw=1, alpha=0.3)
axes[0].axvline(15, color='blue', ls='--', lw=2, alpha=0.5, label='Base 15')
axes[0].axhline(0.05, color='red', ls=':', lw=1, label='5% threshold')
axes[0].set_xlabel(r'Base \(b\)', fontsize=12)
axes[0].set_ylabel(r'Avg dev $|\log_b(r) - \mathrm{round}|$', fontsize=11)
axes[0].set_title(r'Base scan: which \(b\) fits which category?', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 0.4)

# Right: Koide (descriptive) / Справа: Койде (описательно)
axes[1].bar(['Integer\n(0,2,3)', 'Fractions\n(1/32, 1/91)', 'Real masses'],
            [K_int, K_frac, K_real], color=['gray', 'blue', 'green'],
            alpha=0.7, edgecolor='k')
axes[1].axhline(2/3, color='red', ls='--', lw=2, label='Koide = 2/3')
axes[1].set_ylabel(r'Koide ratio \(K\)', fontsize=12)
axes[1].set_title('Koide consistency (descriptive, post-hoc)', fontsize=12)
axes[1].set_ylim(0.64, 0.68)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('10_base_scan.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 10_base_scan.png")

In [ ]:
# ============================================================
# Save to nb10_pvalue_results.txt — single source for all placeholders
# Сохранение в единый файл результатов для всех плейсхолдеров
# ============================================================
lines = [
    "# nb10_pvalue_results.txt",
    f"# Run date: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    "# seed = 42, N_MC = 500",
    "# errors[2] = 0.12 (PDG 2022 for m_tau)",
    "",
    "## DETERMINISTIC SCAN (with MC, stability only)",
    "",
]
for cat_name in ['leptons', 'quarks', 'fermions_9']:
    sorted_res = sorted(results[cat_name], key=lambda x: x[1])
    best_b, best_m, best_s = sorted_res[0]
    lines.append(f"### {cat_name} (best base {best_b}, dev = {best_m:.6f})")
    for rank, (b, m, s) in enumerate(sorted_res[:5], 1):
        lines.append(f"  {rank}. base {b:2d}: dev = {m:.6f} ± {s:.4e}")
    n_below = sum(1 for _, m, _ in results[cat_name] if m < 0.05)
    lines.append(f"  bases < 5%: {n_below}")
    lines.append("")

lines += [
    "## PERMUTATION TEST (deterministic, best-of-39)",
    "",
    f"dev_best39(leptons) = {obs_d_lep:.6f}",
    f"dev_best39(quarks)  = {obs_d_q:.6f}",
    f"observed asym       = {obs_asym:.6f}",
    f"p_abs               = {p_abs_count}/84 = {p_abs_count / 84:.4f}",
    f"rank of leptons     = {rank_lep}/84",
    f"p_asym              = {p_asym_count}/84 = {p_asym_count / 84:.4f}",
    f"p_fixed (base 15)   = {p_fixed_count}/84 = {p_fixed_count / 84:.4f}  (post-selection caveat)",
    f"status              = {status}",
    "",
    "## LOOK-ELSEWHERE",
    "",
    f"top-1 triplet: {{{', '.join(top_names)}}}",
    f"  best base: {top_base}, dev = {top_dev:.6f}",
    f"  ratio (lepton/top): {obs_d_lep / top_dev:.2f}",
]

out = "\n".join(lines)
with open("nb10_pvalue_results.txt", "w") as f:
    f.write(out + "\n")

print("=" * 60)
print("SAVED: nb10_pvalue_results.txt")
print("=" * 60)
print(out)